# NutriMatch - Merge Output Orang A + Orang B

Notebook ini dipakai untuk menyatukan master nutrisi/profil makanan dari Orang A dengan label alergen dari Orang B.

Cara pakai paling aman:

1. Pastikan output Orang B sudah dibuat: `outputs/orang_b/food_allergen_labels.csv`.
2. Clone atau copy repo Orang A ke folder lokal, misalnya `external/orang_a_repo`.
3. Isi `FOOD_MASTER_PATH` dengan path file final Orang A, misalnya `outputs/orang_a/food_master_clean.csv`.
4. Jalankan notebook ini.

Kalau `FOOD_MASTER_PATH = None`, script akan mencari otomatis kandidat food master berdasarkan kolom nama makanan + nutrisi.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'merged'
print('Project root:', PROJECT_ROOT)

## 1. Tentukan Lokasi File Orang A

Isi manual jika file Orang A sudah jelas. Contoh:

```python
FOOD_MASTER_PATH = PROJECT_ROOT / 'external' / 'orang_a_repo' / 'outputs' / 'orang_a' / 'food_master_clean.csv'
```

Jika belum tahu nama filenya, biarkan `None` dan isi `SEARCH_ROOT` ke folder repo Orang A.

In [ ]:
# Default diarahkan ke hasil processed Orang A yang sudah diekstrak di workspace ini.
DEFAULT_ORANG_A_FOOD_MASTER = PROJECT_ROOT / 'ORang A' / 'nutrimatch-capstone-dsA' / 'data' / 'processed' / 'food_master_clean.csv'
FOOD_MASTER_PATH = DEFAULT_ORANG_A_FOOD_MASTER if DEFAULT_ORANG_A_FOOD_MASTER.exists() else None

# Kalau repo Orang A dicopy/clone ke folder tertentu, arahkan ke sana.
# Contoh: SEARCH_ROOT = PROJECT_ROOT / 'external' / 'nutrimatch-capstone-cc26-psu084'
SEARCH_ROOT = PROJECT_ROOT

CONSERVATIVE_UNKNOWN = True

## 2. Jalankan Merge

In [ ]:
from merge_orang_a_b import merge_food_and_allergens

paths = merge_food_and_allergens(
    food_master_path=FOOD_MASTER_PATH,
    search_root=SEARCH_ROOT,
    conservative_unknown=CONSERVATIVE_UNKNOWN,
)

paths

## 3. Cek Ringkasan Merge

In [ ]:
summary = pd.read_csv(OUTPUT_DIR / 'merge_summary.csv')
train_ready = pd.read_csv(OUTPUT_DIR / 'train_ready_dataset.csv')

display(summary)
print('Train-ready shape:', train_ready.shape)
display(train_ready.head())

## 4. Validasi Kolom Final untuk AI Engineer

In [ ]:
required_cols = [
    'food_id', 'food_name', 'food_name_clean',
    'calories_100g', 'protein_100g', 'fat_100g', 'carbohydrate_100g',
    'contains_gluten', 'contains_dairy', 'contains_nuts', 'contains_peanut',
    'contains_seafood', 'contains_egg', 'contains_soy', 'contains_unknown',
    'confidence', 'merge_status'
]

missing = [col for col in required_cols if col not in train_ready.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

display(train_ready['merge_status'].value_counts().to_frame('rows'))
display(train_ready['confidence'].value_counts().to_frame('rows'))

allergen_cols = [col for col in train_ready.columns if col.startswith('contains_')]
display(train_ready[allergen_cols].sum().sort_values(ascending=False).to_frame('positive_rows'))

## 5. Queue Manual Review

Baris `not_matched` dan `contains_unknown=True` perlu dicek manual sebelum dipakai sebagai rekomendasi aman untuk user alergi.

In [ ]:
review_cols = [
    'food_id', 'food_name', 'calories_100g',
    'contains_unknown', 'confidence', 'merge_status', 'label_sources'
]

manual_review = train_ready[
    train_ready['merge_status'].eq('not_matched')
    | train_ready['contains_unknown'].eq(True)
].copy()

print('Manual review rows:', len(manual_review))
display(manual_review[review_cols].head(50))